In [ ]:
!pip install streamlit pyngrok python-docx transformers accelerate bitsandbytes pandas numpy
!pip uninstall pyarrow datasets -y
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 108.9 MB/s eta 0:00:00
Found existing installation: pyarrow 18.1.0
Uninstalling pyarrow-18.1.0:
  Successfully uninstalled pyarrow-18.1.0
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.7 MB/s eta 0:00:00


In [ ]:
import os
import time
import torch
import gc

os.system('pkill -f streamlit')
os.system('pkill -f ngrok')

if os.path.exists('app.py'):
    os.remove('app.py')

print("Готово к созданию app.py")

Готово к созданию app.py


In [ ]:
%%writefile app.py
import streamlit as st
import time
import torch
import gc
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM

st.set_page_config(
    page_title="Автоматическое кодирование интервью",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.title("Автоматическое кодирование интервью")
st.markdown("### Генерация тематических кодов и цитат из текстов интервью")
st.markdown("---")

if 'model_loaded' not in st.session_state:
    st.session_state.model_loaded = False

with st.sidebar:
    st.header("Настройки")

    max_new_tokens = st.slider("Макс. токенов", 512, 4096, 2048, 128)
    num_beams = st.slider("Beam search", 1, 5, 2, 1)
    temperature = st.slider("Температура", 0.0, 1.0, 0.0, 0.1)

    st.markdown("---")
    st.info("Приложение кодирует интервью с помощью Qwen2.5-7B-Instruct")

@st.cache_resource
def load_model():
    with st.spinner("Загрузка модели..."):
        base_model = "Qwen/Qwen2.5-7B-Instruct"

        tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16
        )

        model = AutoModelForCausalLM.from_pretrained(
            base_model,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.bfloat16
        )

        model.eval()
        st.success("Модель загружена!")
        return model, tokenizer

def build_prompt(topic, transcript):
    return f"""Ты - эксперт по анализу интервью. Выдели тематические коды и цитаты.

Формат вывода:
**Общий код 1: <название>**
"<цитата>" - **<конкретный код> (Конкретный код)**

Тема: {topic}

Текст интервью:
{transcript}

Ответ на русском языке, только коды и цитаты:"""

def generate(model, tokenizer, prompt, max_tokens, beams, temp):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=10000)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=temp > 0,
            temperature=temp if temp > 0 else 1.0,
            num_beams=beams if temp == 0 else 1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def main():
    if not st.session_state.model_loaded:
        try:
            model, tokenizer = load_model()
            st.session_state.model = model
            st.session_state.tokenizer = tokenizer
            st.session_state.model_loaded = True
        except Exception as e:
            st.error(f"Ошибка: {e}")
            st.stop()
    else:
        model = st.session_state.model
        tokenizer = st.session_state.tokenizer

    st.header("Кодирование интервью")

    with st.form("main_form"):
        topic = st.text_area("Тема интервью", height=100)
        transcript = st.text_area("Текст интервью", height=400)
        submitted = st.form_submit_button("Выполнить кодирование", type="primary")

    if submitted:
        if not topic or not transcript:
            st.error("Заполните все поля!")
        else:
            with st.spinner("Генерация..."):
                prompt = build_prompt(topic, transcript)
                result = generate(model, tokenizer, prompt, max_new_tokens, num_beams, temperature)

                st.success("Готово!")
                st.markdown("### Результат")
                st.markdown(result)

                st.download_button("Скачать TXT", result, f"coding_{int(time.time())}.txt")

if __name__ == "__main__":
    main()

Writing app.py


In [ ]:
import os
import time

os.system('streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true &')
time.sleep(5)

from pyngrok import ngrok
ngrok.kill()
ngrok.set_auth_token("код из ngrok")
public_url = ngrok.connect(8501)

print(f"\nПубличный URL: {public_url}")
print("Откройте ссылку в браузере")


Публичный URL: NgrokTunnel: "https://cassette-sequester-video.ngrok-free.dev" -> "http://localhost:8501"
Откройте ссылку в браузере
